# 03 — End-to-End Similarity Demo

Demonstrates the full **content-based music retrieval** pipeline:

1. Generate synthetic tracks with known acoustic properties
2. Extract deep embeddings via the CNN encoder
3. Index all tracks in FAISS
4. Query the index and inspect results
5. Visualise the embedding space with t-SNE

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.mir.models.encoder import MIRModel, EmbeddingInferencer
from src.mir.features.spectrogram import compute_mel_spectrogram, mel_to_tensor
from src.mir.search.faiss_index import FaissSearchEngine
from src.mir.utils.visualization import plot_similarity_heatmap, plot_embedding_scatter

np.random.seed(42)
torch.manual_seed(42)
%matplotlib inline
print('Ready!')

## 1. Create Synthetic Track Library

3 'genre clusters': low-freq bass tracks, high-freq treble tracks, mid-range tracks.

In [ ]:
SR = 22050
DUR = 5.0
t = np.linspace(0, DUR, int(SR * DUR))

def make_track(freqs, amplitudes, noise=0.05):
    y = sum(a * np.sin(2 * np.pi * f * t) for f, a in zip(freqs, amplitudes))
    y += noise * np.random.randn(len(t))
    return y.astype(np.float32)

tracks = {}
# Bass-heavy cluster
for i in range(6):
    tracks[f'bass_{i}'] = make_track(
        [60 + i*5, 120 + i*5], [0.6, 0.3], noise=0.03 + i*0.01
    )
# Treble-heavy cluster  
for i in range(6):
    tracks[f'treble_{i}'] = make_track(
        [2000 + i*100, 4000 + i*100], [0.5, 0.4], noise=0.04
    )
# Mid-range cluster
for i in range(6):
    tracks[f'mid_{i}'] = make_track(
        [440 + i*20, 880 + i*20], [0.5, 0.35], noise=0.05
    )

print(f'Created {len(tracks)} synthetic tracks in 3 clusters')

## 2. Extract Embeddings

In [ ]:
model = MIRModel()
inferencer = EmbeddingInferencer(model, device='cpu')

embeddings = {}
for track_id, y in tracks.items():
    mel = compute_mel_spectrogram(y, SR)
    tensor = mel_to_tensor(mel)
    emb = inferencer.embed(tensor)
    embeddings[track_id] = emb.numpy()

emb_matrix = np.array(list(embeddings.values()))   # (18, 256)
print(f'Embedding matrix: {emb_matrix.shape}')
print(f'L2 norms (should all be ~1.0): {np.linalg.norm(emb_matrix, axis=1).round(4)}')

## 3. Build FAISS Index

In [ ]:
engine = FaissSearchEngine(dim=256, nlist=4, M=0, nprobe=4)

# Train on all embeddings
engine.train(emb_matrix)

# Add all tracks
engine.add(emb_matrix, list(embeddings.keys()))
print(f'Index size: {len(engine)} vectors')

## 4. Similarity Search

In [ ]:
# Query with bass_0 — should return other bass_ tracks first
query_id = 'bass_0'
query_emb = embeddings[query_id]

results = engine.search(query_emb, top_k=8, exclude_ids=[query_id])

print(f'Query: "{query_id}"')
print(f'{"Rank":<6} {"Track ID":<15} {"Score":<10}')
print('-' * 35)
for r in results:
    cluster = r.track_id.split('_')[0]
    match = '✅' if cluster == 'bass' else '❌'
    print(f'{r.rank:<6} {r.track_id:<15} {r.score:.4f}  {match}')

## 5. Similarity Heatmap

In [ ]:
# Show pairwise similarity for first 12 tracks (4 per cluster)
subset_ids = list(embeddings.keys())[:12]
subset_embs = np.array([embeddings[k] for k in subset_ids])

fig = plot_similarity_heatmap(subset_embs, subset_ids,
                               title='Pairwise Cosine Similarity')
plt.show()

## 6. t-SNE Embedding Visualisation

Should show 3 well-separated clusters.

In [ ]:
labels = [tid.split('_')[0] for tid in embeddings.keys()]

fig = plot_embedding_scatter(
    emb_matrix, labels,
    method='tsne',
    title='Track Embedding Space'
)
plt.show()
print('3 distinct clusters visible = embeddings capture tonal character')

## 7. Genre Classification Demo

In [ ]:
for track_id in ['bass_0', 'treble_0', 'mid_0']:
    y = tracks[track_id]
    mel = compute_mel_spectrogram(y, SR)
    tensor = mel_to_tensor(mel)
    result = inferencer.classify_genre(tensor)
    print(f'{track_id:12s} → genre: {result["label"]:12s} (conf: {result["confidence"]:.2f})')